In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance_segmented_2004.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    Species  = as.character(Species),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_abundance_std","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_abundance_std, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, zone, Species, HYBAS_ID, SiteID
    )
  )

vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

core_vars <- c("mean_temp", "sen_temp")

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

fixed_form <- sen_abundance_std ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | Species) + (1 | HYBAS_ID) + (1 | SiteID) + (1 | Protocol)
)
zones <- c("All","Cold","Intermediate","Warm")
vif_threshold <- 10
get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else {
    vif_raw
  }
}

is_high_vif <- function(v, vif_vals) {

  if (v %in% core_vars) return(FALSE)

  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  if (length(terms) == 0) return(FALSE)

  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p){
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}
effects_all     <- list()
model_stats     <- list()
model_summaries <- list()
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running zone:", z, "\n")

  # ---- VIF ----
  vif_vals <- get_vif_vals(dat)
  high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
  names(high_vif) <- vars

  # ---- Model ----
  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # ---- Save summary ----
  model_summaries[[z]] <- summary(m)

  # ---- Fit stats ----
  r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
  model_stats[[z]] <- data.frame(
    Zone = z,
    AIC = AIC(m),
    BIC = BIC(m),
    R2_marginal = r2[1],
    R2_conditional = r2[2]
  )

  cf <- coef(summary(m))

  for (v in vars) {

    if (isTRUE(high_vif[[v]])) {
      effects_all[[length(effects_all)+1]] <- data.frame(
        Zone = z, Factor = v,
        Early_Effect = NA, Early_CI95_L = NA, Early_CI95_U = NA,
        Early_p = NA, Early_sig = "",
        Late_Effect = NA, Late_CI95_L = NA, Late_CI95_U = NA,
        Late_p = NA, Late_sig = "",
        Delta = NA, Delta_p = NA
      )
      next
    }

    vz <- paste0(v, "_z")

    # ---- Early AME ----
    ame_e <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 0)
    )

    # ---- Late AME ----
    ame_l <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 1)
    )

    b_e <- ame_e$estimate[1]
    p_e <- ame_e$p.value[1]
    ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])

    b_l <- ame_l$estimate[1]
    p_l <- ame_l$p.value[1]
    ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])

    # ---- Delta = interaction ----
    inter1 <- paste0(vz, ":Period_bin")
    inter2 <- paste0("Period_bin:", vz)
    inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

    b_d <- b_l - b_e
    p_d <- if (inter %in% rownames(cf)) cf[inter, "Pr(>|t|)"] else NA

    effects_all[[length(effects_all)+1]] <- data.frame(
      Zone = z, Factor = v,

      Early_Effect = b_e,
      Early_CI95_L = ci_e[1], Early_CI95_U = ci_e[2],
      Early_p = p_e, Early_sig = sig_label(p_e),

      Late_Effect = b_l,
      Late_CI95_L = ci_l[1], Late_CI95_U = ci_l[2],
      Late_p = p_l, Late_sig = sig_label(p_l),

      Delta = b_d, Delta_p = p_d
    )
  }
}

# =========================================================
# Output: AME & model stats
# =========================================================
write_csv(
  bind_rows(effects_all),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/Period_effects_by_zone_with_CI_Abundance_AME.csv"
)

write_csv(
  bind_rows(model_stats),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/Model_fit_stats_Abundance.csv"
)

# =========================================================
# Output: model summaries ((Dataset S3))
# =========================================================
sink("C:/Users/Lenovo/Phase-specific modelling of abundance trends/Abundance_model_summaries_by_zone.txt")

for (z in names(model_summaries)) {
  cat(
    "\n=====================================================\n",
    "Zone:", z, "\n",
    "=====================================================\n\n"
  )
  print(model_summaries[[z]])
}

sink()

Warning message:
"package 'lme4' was built under R version 4.5.2"
Loading required package: Matrix

Warning message:
"package 'lmerTest' was built under R version 4.5.2"

Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'readr' was built under R version 4.5.2"
Warning message:
"package 'car' was built under R version 4.5.2"
Loading required package: carData

Warning message:
"package 'carData' was built under R version 4.5.2"

Attaching package: 'car'


The following object is masked from 'package:dplyr':

    recode


Warning message:
"package 'MuMIn' was built under R version 4.5.2"
Wa

Running zone: All 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or 

Running zone: Cold 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surpris

Running zone: Intermediate 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surpris

Running zone: Warm 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or 

In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(spdep)


df <- read_csv(
  "D:/NC/Data/rivernet/inputdata/Abundance_segmented_2004.csv"
)

df <- df %>%
  mutate(
    SiteID = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )


scale_cols <- c(
  "sen_temp","HFP_period","mean_temp",
  "mean_Q","mean_salinity","mean_organic","elevation"
)
for (v in scale_cols) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}


form <- sen_abundance_std ~
  Period_bin +
  sen_temp_z + HFP_period_z + mean_temp_z + mean_Q_z +
  mean_salinity_z + mean_organic_z + elevation_z +
  sen_temp_z:Period_bin +
  HFP_period_z:Period_bin +
  mean_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_salinity_z:Period_bin +
  mean_organic_z:Period_bin +
  elevation_z:Period_bin +
  (1 | HYBAS_ID) + (1 | Protocol) + (1 | SiteID) + (1 | Species)

zones <- c("All","Cold","Intermediate","Warm")

# ===============================
# 4. Moran’s I (Dataset S3)
# ===============================
out <- list()

for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  m <- lmer(form, data = dat, REML = TRUE)
  mf <- model.frame(m)
  mf$resid <- resid(m)
  dat_used <- dat[as.numeric(rownames(mf)), ]

  mf$SiteID    <- dat_used$SiteID
  mf$Longitude <- dat_used$Longitude
  mf$Latitude  <- dat_used$Latitude
  site_res <- mf %>%
    group_by(SiteID) %>%
    summarise(
      resid = mean(resid, na.rm = TRUE),
      Longitude = first(Longitude),
      Latitude  = first(Latitude),
      .groups = "drop"
    )

  coords <- as.matrix(site_res[, c("Longitude", "Latitude")])

  knn <- knearneigh(coords, k = 10)
  nb  <- knn2nb(knn)
  lw  <- nb2listw(nb, style = "W", zero.policy = TRUE)

  mi <- moran.test(
    site_res$resid,
    lw,
    randomisation = TRUE,
    zero.policy = TRUE
  )

  out[[z]] <- data.frame(
    Zone = z,
    N_sites = nrow(site_res),
    Moran_I = unname(mi$estimate["Moran I statistic"]),
    P_value = mi$p.value
  )
}

df_moran <- bind_rows(out)

write_csv(
  df_moran,
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/MoranI_residuals_by_zone_SITE_Abundance.csv"
)
print(df_moran)

Warning message:
"package 'spdep' was built under R version 4.5.2"
Loading required package: spData

Warning message:
"package 'spData' was built under R version 4.5.2"
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='source')`

Loading required package: sf

Linking to GEOS 3.13.1, GDAL 3.11.0, PROJ 9.6.0; sf_use_s2() is TRUE

Rows: 51194 Columns: 22
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): SiteID, Species, period, Protocol, zone
dbl (17): sen_abundance_std, p_abundance_std, n_years, mean_salinity, mean_o...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model failed to converge with max|grad| = 0.00276055 (tol = 0.002,

          Zone N_sites     Moran_I      P_value
1          All    3340  0.05880517 2.391632e-16
2         Cold     661  0.05220920 4.032325e-04
3 Intermediate    1951  0.04818447 1.371085e-07
4         Warm     728 -0.01727075 8.469146e-01


In [ ]:
# =========================================================
#  random-structure comparison (ML) (Dataset S8)
# =========================================================
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(MuMIn)
library(marginaleffects)
library(tibble)


`%||%` <- function(x, y) if (length(x) == 0 || all(is.na(x))) y else x

sig_label <- function(p){
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance_segmented_2004.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    Species  = as.character(Species),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_abundance_std","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>% filter(
  complete.cases(
    sen_abundance_std, sen_temp, HFP_period,
    mean_temp, mean_Q, mean_salinity,
    mean_organic, elevation,
    Protocol, zone, Species, HYBAS_ID, SiteID
  )
)

vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

for (v in vars){
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

fixed_form <- sen_abundance_std ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

zones <- c("All","Cold","Intermediate","Warm")

specs <- list(
  M1 = ". ~ . + (1|Species) + (1|HYBAS_ID) + (1|Protocol) ",
  M2 = ". ~ . + (1|Species) + (1|HYBAS_ID) + (1|Protocol) + (1|SiteID)",
  M3 = ". ~ . + (1|Species) + (1|HYBAS_ID) + C(Protocol)",
  M4 = ". ~ . + (1|Species) + (1|HYBAS_ID) + C(Protocol) + (1|SiteID)",  
  M5 = ". ~ . + (1|Species) + (1|Protocol) + (1|SiteID)",
  M6 = ". ~ . + (1|Species) + C(Protocol) + (1|SiteID)"
)

get_all_terms <- function(m){
  coef(summary(m)) %>%
    as.data.frame() %>%
    rownames_to_column("Term") %>%
    mutate(
      p = `Pr(>|t|)`,
      sig = sapply(p, sig_label),
      Effect_type = case_when(
        grepl(":Period_bin", Term) | grepl("Period_bin:", Term) ~ "Delta",
        Term == "Period_bin" ~ "Period",
        TRUE ~ "Main"
      ),
      Delta_var = ifelse(
        Effect_type == "Delta",
        gsub(":Period_bin|Period_bin:", "", Term),
        NA
      )
    )
}

get_ame_all <- function(m, dat){
  out <- list()

  for (v in vars){
    vz <- paste0(v, "_z")

    a0 <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 0)
    )
    a1 <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 1)
    )

    out[[v]] <- data.frame(
      Variable = v,
      Early_Effect = a0$estimate[1],
      Early_p = a0$p.value[1],
      Late_Effect  = a1$estimate[1],
      Late_p  = a1$p.value[1],
      Delta = a1$estimate[1] - a0$estimate[1]
    )
  }

  bind_rows(out)
}

fit_rows  <- list()
coef_rows <- list()
ame_rows  <- list()

for (z in zones){

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("\n========================\nZone:", z, " N=", nrow(dat), "\n")

  for (nm in names(specs)){

    form <- update(fixed_form, as.formula(specs[[nm]]))

    m <- tryCatch(
      lmer(
        form, data = dat, REML = FALSE,
        control = lmerControl(
          optimizer = "bobyqa",
          optCtrl = list(maxfun = 2e5)
        )
      ),
      error = function(e) NULL
    )
    if (is.null(m)) next
    r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
    vc <- as.data.frame(VarCorr(m))

    fit_rows[[length(fit_rows)+1]] <- data.frame(
      Zone = z, Model = nm,
      N = nrow(dat),
      AIC = AIC(m), BIC = BIC(m),
      R2_marginal = r2[1], R2_conditional = r2[2],
      Singular = isSingular(m),
      Var_Species  = vc$vcov[vc$grp=="Species"][1] %||% NA,
      Var_HYBAS    = vc$vcov[vc$grp=="HYBAS_ID"][1] %||% NA,
      Var_SiteID   = vc$vcov[vc$grp=="SiteID"][1] %||% NA,
      Var_Residual = vc$vcov[vc$grp=="Residual"][1] %||% NA
    )
    cf_all <- get_all_terms(m) %>%
      mutate(Zone = z, Model = nm)
    coef_rows[[length(coef_rows)+1]] <- cf_all
    ame <- get_ame_all(m, dat) %>%
      mutate(Zone = z, Model = nm)
    ame_rows[[length(ame_rows)+1]] <- ame
  }
}
write_csv(
  bind_rows(fit_rows),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/SensA_ModelCompare_ML_Abundance_CProtocol.csv"
)

write_csv(
  bind_rows(coef_rows),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/SensA_AllFixedEffects_ML_Abundance_CProtocol.csv"
)

write_csv(
  bind_rows(ame_rows),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/SensA_AME_allvars_ML_Abundance_CProtocol.csv"
)

Rows: 51194 Columns: 22
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): SiteID, Species, period, Protocol, zone
dbl (17): sen_abundance_std, p_abundance_std, n_years, mean_salinity, mean_o...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



Zone: All  N= 29908 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa


Zone: Cold  N= 3746 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 


Zone: Intermediate  N= 15346 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa


Zone: Warm  N= 10816 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

In [ ]:
# =========================================================
# Sensitivity S1: Outlier influence (trim top X% abs residuals) (Dataset S10)
# =========================================================
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================
# User settings
# =========================
trim_prop <- 0.01      # top 1% |resid|
zones <- c("All","Cold","Intermediate","Warm")

# =========================
# Read data (ABUNDANCE)
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance_segmented_2004.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    Species  = as.character(Species),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    period   = as.character(period),
    Period_bin = ifelse(period == "2005_2018", 1, 0)
  )

num_cols <- c(
  "sen_abundance_std","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_abundance_std, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, zone, Species, HYBAS_ID, SiteID
    )
  )

# =========================
# Global Z-score
# =========================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)
for (v in vars) df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]

# =========================
# Model
# =========================
fixed_form <- sen_abundance_std ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form_main <- update(
  fixed_form,
  . ~ . + (1|HYBAS_ID) + (1|Protocol) + (1|SiteID) + (1 | Species)
)

# =========================
# Helpers
# =========================

# ---- extract Delta p + CI from interaction term ----
get_delta_stats <- function(m, v){
  vz <- paste0(v, "_z")
  cf <- coef(summary(m))

  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  if (!inter %in% rownames(cf)) {
    return(c(NA, NA, NA))
  }

  est <- cf[inter, "Estimate"]
  se  <- cf[inter, "Std. Error"]
  df  <- cf[inter, "df"]
  p   <- cf[inter, "Pr(>|t|)"]

  tcrit <- qt(0.975, df = df)

  c(
    Delta_p = p,
    Delta_CI_L = est - tcrit * se,
    Delta_CI_U = est + tcrit * se
  )
}

# ---- AME + Delta ----
get_ame_allvars <- function(m, dat, z, tag){
  out <- list()

  for (v in vars){
    vz <- paste0(v, "_z")

    a0 <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 0)
    )
    a1 <- avg_slopes(
      m, variables = vz,
      newdata = dat %>% mutate(Period_bin = 1)
    )

    delta_stats <- get_delta_stats(m, v)

    out[[v]] <- data.frame(
      Zone = z,
      Tag = tag,
      Factor = v,

      Early_Effect = a0$estimate[1],
      Early_CI_L = a0$conf.low[1],
      Early_CI_U = a0$conf.high[1],
      Early_p = a0$p.value[1],

      Late_Effect = a1$estimate[1],
      Late_CI_L = a1$conf.low[1],
      Late_CI_U = a1$conf.high[1],
      Late_p = a1$p.value[1],

      Delta = a1$estimate[1] - a0$estimate[1],
      Delta_p = delta_stats["Delta_p"],
      Delta_CI_L = delta_stats["Delta_CI_L"],
      Delta_CI_U = delta_stats["Delta_CI_U"]
    )
  }

  bind_rows(out)
}

# =========================
# Run S1
# =========================
sum_rows <- list()
ame_rows <- list()

for (z in zones){

  dat0 <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat0) < 50) next

  cat("\nRunning S1 | Zone:", z, " N=", nrow(dat0), "\n")

  # ---- full model ----
  m_full <- lmer(
    form_main, data = dat0, REML = TRUE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # ---- trim top |residual| ----
  r <- residuals(m_full, type = "pearson")
  cut <- quantile(abs(r), 1 - trim_prop, na.rm = TRUE)
  dat1 <- dat0[abs(r) <= cut, , drop = FALSE]

  # ---- trimmed model ----
  m_trim <- lmer(
    form_main, data = dat1, REML = TRUE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # ---- fit stats ----
  r2_full <- suppressWarnings(MuMIn::r.squaredGLMM(m_full))
  r2_trim <- suppressWarnings(MuMIn::r.squaredGLMM(m_trim))

  sum_rows[[length(sum_rows)+1]] <- data.frame(
    Zone = z,
    N_full = nrow(dat0),
    N_trim = nrow(dat1),
    Trim_prop = 1 - nrow(dat1) / nrow(dat0),
    R2m_full = r2_full[1],
    R2c_full = r2_full[2],
    R2m_trim = r2_trim[1],
    R2c_trim = r2_trim[2],
    AIC_full = AIC(m_full),
    AIC_trim = AIC(m_trim)
  )

  # ---- AME outputs ----
  ame_rows[[length(ame_rows)+1]] <- get_ame_allvars(m_full, dat0, z, "Full")
  ame_rows[[length(ame_rows)+1]] <- get_ame_allvars(m_trim, dat1, z, "Trim")
}

# =========================
# Output
# =========================
write_csv(
  bind_rows(sum_rows),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/SensS1_OutlierTrim_summary_Abundance.csv"
)

write_csv(
  bind_rows(ame_rows),
  "C:/Users/Lenovo/Phase-specific modelling of abundance trends/SensS1_OutlierTrim_AME_allvars_Abundance.csv"
)


Rows: 51194 Columns: 22
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): SiteID, Species, period, Protocol, zone
dbl (17): sen_abundance_std, p_abundance_std, n_years, mean_salinity, mean_o...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



Running S1 | Zone: All  N= 29908 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa


Running S1 | Zone: Cold  N= 3746 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 


Running S1 | Zone: Intermediate  N= 15346 


boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some


Running S1 | Zone: Warm  N= 10816 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================================================
# Read data  2003 turn point
# =========================================================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance_segmented_2003.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    Species  = as.character(Species),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    Period_bin = ifelse(period == "2004_2018", 1, 0)
  )

num_cols <- c(
  "sen_abundance_std","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_abundance_std, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, Species, HYBAS_ID, SiteID
    )
  )

# =========================================================
# Variables
# =========================================================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

core_vars <- c("mean_temp", "sen_temp")

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

# =========================================================
# Model formula
# =========================================================
fixed_form <- sen_abundance_std ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | Species) + (1 | HYBAS_ID) + (1 | SiteID) + (1 | Protocol)
)

# =========================================================
# VIF helpers
# =========================================================
vif_threshold <- 10

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else vif_raw
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)

  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  if (length(terms) == 0) return(FALSE)

  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p){
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================================================
# Run model
# =========================================================
effects_all     <- list()
model_stats     <- list()
model_summaries <- list()

z <- "All"
dat <- df

cat("Running zone:", z, "\n")

# ---- VIF ----
vif_vals <- get_vif_vals(dat)
high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
names(high_vif) <- vars

# ---- Model ----
m <- lmer(
  form, data = dat, REML = TRUE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# ---- Save summary ----
model_summaries[[z]] <- summary(m)

# ---- Fit stats ----
r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
model_stats[[z]] <- data.frame(
  Zone = z,
  AIC = AIC(m),
  BIC = BIC(m),
  R2_marginal = r2[1],
  R2_conditional = r2[2]
)

cf <- coef(summary(m))

for (v in vars) {

  if (isTRUE(high_vif[[v]])) {
    effects_all[[length(effects_all)+1]] <- data.frame(
      Zone = z, Factor = v,
      Early_Effect = NA, Early_CI95_L = NA, Early_CI95_U = NA,
      Early_p = NA, Early_sig = "",
      Late_Effect = NA, Late_CI95_L = NA, Late_CI95_U = NA,
      Late_p = NA, Late_sig = "",
      Delta = NA, Delta_p = NA
    )
    next
  }

  vz <- paste0(v, "_z")

  ame_e <- avg_slopes(
    m, variables = vz,
    newdata = dat %>% mutate(Period_bin = 0)
  )

  ame_l <- avg_slopes(
    m, variables = vz,
    newdata = dat %>% mutate(Period_bin = 1)
  )

  b_e <- ame_e$estimate[1]
  p_e <- ame_e$p.value[1]
  ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])

  b_l <- ame_l$estimate[1]
  p_l <- ame_l$p.value[1]
  ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])

  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  b_d <- b_l - b_e
  p_d <- if (inter %in% rownames(cf)) cf[inter, "Pr(>|t|)"] else NA

  effects_all[[length(effects_all)+1]] <- data.frame(
    Zone = z, Factor = v,

    Early_Effect = b_e,
    Early_CI95_L = ci_e[1], Early_CI95_U = ci_e[2],
    Early_p = p_e, Early_sig = sig_label(p_e),

    Late_Effect = b_l,
    Late_CI95_L = ci_l[1], Late_CI95_U = ci_l[2],
    Late_p = p_l, Late_sig = sig_label(p_l),

    Delta = b_d, Delta_p = p_d
  )
}

# =========================================================
# Output
# =========================================================
write_csv(
  bind_rows(effects_all),
  "C:/Users/Lenovo/Phase-specific modelling of 2003/Period_effects_All_Abundance_AME.csv"
)

write_csv(
  bind_rows(model_stats),
  "C:/Users/Lenovo/Phase-specific modelling of 2003/Model_fit_stats_Abundance_All.csv"
)

sink("C:/Users/Lenovo/Phase-specific modelling of 2003/Abundance_model_summaries_by_zone.txt")
cat("Zone: All\n\n")
print(model_summaries[["All"]])
sink()


Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 51194 Columns: 32
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Species, period, Protocol, zone, UnitAbundance, HFP_class
dbl (25): seg_id_nat, start_year, end_year, abun_n_obs, temp_n_obs, q_n_obs,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running zone: All 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or 

In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================================================
# Read data 2005 turn point
# =========================================================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Abundance_segmented_2005.csv") %>%
  mutate(
    SiteID   = as.character(SiteID),
    Species  = as.character(Species),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone),
    Period_bin = ifelse(period == "2006_2018", 1, 0)
  )

num_cols <- c(
  "sen_abundance_std","sen_temp","HFP_period",
  "mean_temp","mean_Q","mean_salinity",
  "mean_organic","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      sen_abundance_std, sen_temp, HFP_period,
      mean_temp, mean_Q, mean_salinity,
      mean_organic, elevation,
      Protocol, Species, HYBAS_ID, SiteID
    )
  )

# =========================================================
# Variables
# =========================================================
vars <- c(
  "mean_temp","sen_temp","mean_Q",
  "mean_organic","mean_salinity",
  "HFP_period","elevation"
)

core_vars <- c("mean_temp", "sen_temp")

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[,1]
}

# =========================================================
# Model formula
# =========================================================
fixed_form <- sen_abundance_std ~
  Period_bin +
  mean_temp_z + sen_temp_z + mean_Q_z +
  mean_organic_z + mean_salinity_z +
  HFP_period_z + elevation_z +
  mean_temp_z:Period_bin +
  sen_temp_z:Period_bin +
  mean_Q_z:Period_bin +
  mean_organic_z:Period_bin +
  mean_salinity_z:Period_bin +
  HFP_period_z:Period_bin +
  elevation_z:Period_bin

form <- update(
  fixed_form,
  . ~ . + (1 | Species) + (1 | HYBAS_ID) + (1 | SiteID) + (1 | Protocol)
)

# =========================================================
# VIF helpers
# =========================================================
vif_threshold <- 10

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else vif_raw
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)

  terms <- c(
    paste0(v, "_z"),
    paste0(v, "_z:Period_bin"),
    paste0("Period_bin:", v, "_z")
  )
  terms <- terms[terms %in% names(vif_vals)]
  if (length(terms) == 0) return(FALSE)

  any(vif_vals[terms] >= vif_threshold)
}

sig_label <- function(p){
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================================================
# Run model
# =========================================================
effects_all     <- list()
model_stats     <- list()
model_summaries <- list()

z <- "All"
dat <- df

cat("Running zone:", z, "\n")

# ---- VIF ----
vif_vals <- get_vif_vals(dat)
high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
names(high_vif) <- vars

# ---- Model ----
m <- lmer(
  form, data = dat, REML = TRUE,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

# ---- Save summary ----
model_summaries[[z]] <- summary(m)

# ---- Fit stats ----
r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
model_stats[[z]] <- data.frame(
  Zone = z,
  AIC = AIC(m),
  BIC = BIC(m),
  R2_marginal = r2[1],
  R2_conditional = r2[2]
)

cf <- coef(summary(m))

for (v in vars) {

  if (isTRUE(high_vif[[v]])) {
    effects_all[[length(effects_all)+1]] <- data.frame(
      Zone = z, Factor = v,
      Early_Effect = NA, Early_CI95_L = NA, Early_CI95_U = NA,
      Early_p = NA, Early_sig = "",
      Late_Effect = NA, Late_CI95_L = NA, Late_CI95_U = NA,
      Late_p = NA, Late_sig = "",
      Delta = NA, Delta_p = NA
    )
    next
  }

  vz <- paste0(v, "_z")

  ame_e <- avg_slopes(
    m, variables = vz,
    newdata = dat %>% mutate(Period_bin = 0)
  )

  ame_l <- avg_slopes(
    m, variables = vz,
    newdata = dat %>% mutate(Period_bin = 1)
  )

  b_e <- ame_e$estimate[1]
  p_e <- ame_e$p.value[1]
  ci_e <- c(ame_e$conf.low[1], ame_e$conf.high[1])

  b_l <- ame_l$estimate[1]
  p_l <- ame_l$p.value[1]
  ci_l <- c(ame_l$conf.low[1], ame_l$conf.high[1])

  inter1 <- paste0(vz, ":Period_bin")
  inter2 <- paste0("Period_bin:", vz)
  inter  <- if (inter1 %in% rownames(cf)) inter1 else inter2

  b_d <- b_l - b_e
  p_d <- if (inter %in% rownames(cf)) cf[inter, "Pr(>|t|)"] else NA

  effects_all[[length(effects_all)+1]] <- data.frame(
    Zone = z, Factor = v,

    Early_Effect = b_e,
    Early_CI95_L = ci_e[1], Early_CI95_U = ci_e[2],
    Early_p = p_e, Early_sig = sig_label(p_e),

    Late_Effect = b_l,
    Late_CI95_L = ci_l[1], Late_CI95_U = ci_l[2],
    Late_p = p_l, Late_sig = sig_label(p_l),

    Delta = b_d, Delta_p = p_d
  )
}

# =========================================================
# Output
# =========================================================
write_csv(
  bind_rows(effects_all),
  "C:/Users/Lenovo/Phase-specific modelling of 2005/Period_effects_All_Abundance_AME.csv"
)

write_csv(
  bind_rows(model_stats),
  "C:/Users/Lenovo/Phase-specific modelling of 2005/Model_fit_stats_Abundance_All.csv"
)

sink("C:/Users/Lenovo/Phase-specific modelling of 2005/Abundance_model_summaries_by_zone.txt")
cat("Zone: All\n\n")
print(model_summaries[["All"]])
sink()


Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 51194 Columns: 32
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Species, period, Protocol, zone, UnitAbundance, HFP_class
dbl (25): seg_id_nat, start_year, end_year, abun_n_obs, temp_n_obs, q_n_obs,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running zone: All 


there are higher-order terms (interactions) in this model
consider setting type = 'predictor'; see ?vif

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or 